In [ ]:
# 04_evaluation_explicabilite.ipynb
!pip install shap lime 
import shap
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from lime.lime_tabular import LimeTabularExplainer
import seaborn as sns
from sklearn.inspection import permutation_importance

# Entraînement final sur tout le jeu d'entraînement
xgb.fit(X_selected, y_res)

# Prédictions sur un jeu de test (à définir)
# X_test_selected, y_test = ...

# Rapport de classification
y_pred = xgb.predict(X_test_selected)
print(classification_report(y_test, y_pred))

# Matrice de confusion
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True)
plt.clf()
plt.title('Matrice de confusion')
plt.show()

# Analyse des erreurs
misclassified = X_test_selected[y_pred != y_test]
print("Exemples mal classés :", misclassified)

# SHAP (global et local)
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test_selected)
shap.summary_plot(shap_values, X_test_selected, feature_names=selected_features)
shap.force_plot(explainer.expected_value, shap_values[0], X_test_selected[0], feature_names=selected_features)

# LIME (local)
lime_explainer = LimeTabularExplainer(X_selected, feature_names=selected_features, class_names=['Sain', 'Malade'], discretize_continuous=True)
exp = lime_explainer.explain_instance(X_test_selected[0], xgb.predict_proba)
exp.show_in_notebook()

# Permutation importance
result = permutation_importance(xgb, X_test_selected, y_test, n_repeats=10, random_state=42)
plt.clf()
plt.barh(selected_features, result.importances_mean)
plt.title('Permutation Importance')
plt.show()